# Radiate traces for Metabolites from stIds


In [10]:
import os
import warnings
from pathlib import PurePosixPath, Path
import pandas as pd



# Ignore warnings
warnings.filterwarnings('ignore')

In [2]:
import dotenv

# Load environment variables from .env file
dotenv.load_dotenv()

# Directory where to look for input data
input_dir = PurePosixPath('input')
print(Path(input_dir).resolve())

# Directory where to output results
output_dir = PurePosixPath('output')
os.makedirs(output_dir, exist_ok=True)

/Users/robin/dev/lifelike-gds/notebooks/reactome/input


In [8]:
source_name = 'abnormal_metabolites'
source_file = 'metabolite_stIds.csv'

nodes_select_file = 'Radiate_analysis_for_abnormal_metabolites_selection.xlsx'

## Get Source and Target Nodes

In [11]:
# Read source file
source_path = input_dir / source_file

df = pd.read_csv(source_path)
st_ids = df['stId'].tolist()

In [21]:
# Read node selection file
def get_selected_node_ids(filename, sheet_name, select_column='select'):
    file_path = input_dir / filename
    df = pd.read_excel(file_path, sheet_name)
    mydf = df[df[select_column] == 1]
    selected_nodes = [id for id in mydf['stId']]
    print(f'selected {sheet_name} nodes:\n', selected_nodes)
    return selected_nodes

forward_selection = get_selected_node_ids(nodes_select_file, 'pageranks')
reverse_selection = get_selected_node_ids(nodes_select_file, 'reverse pageranks')

selected pageranks nodes:
 ['R-HSA-2173761', 'R-ALL-198816', 'R-HSA-400543', 'R-HSA-2453678', 'R-ALL-113557', 'R-HSA-1234149', 'R-HSA-1234131', 'R-HSA-1234129', 'R-HSA-74841', 'R-HSA-8964323']
selected reverse pageranks nodes:
 ['R-HSA-5690911', 'R-ALL-29450', 'R-ALL-34979', 'R-HSA-70904', 'R-HSA-74841', 'R-HSA-5690899', 'R-HSA-70900', 'R-HSA-71679', 'R-HSA-70897', 'R-HSA-433093', 'R-HSA-433135', 'R-HSA-372445', 'R-HSA-975606', 'R-HSA-6783219', 'R-HSA-174364', 'R-HSA-2429687']


## Connect to graph database, retrieve source and selected tracing nodes

In [22]:
from lifelike_gds.arango_network.reactome import Reactome, ReactomeDB
from lifelike_gds.arango_network.radiate_trace import RadiateTrace

database = ReactomeDB(os.getenv('ARANGO_DATABASE', 'reactome'),
                      os.getenv('ARANGO_URI', 'localhost'), 
                      os.getenv('ARANGO_USER', 'root'), 
                      os.getenv('ARANGO_PASSWORD', ''))

source_nodes = database.get_nodes_by_attr(attr_values=st_ids, attr_name="stId")
print(f"Found source nodes: {len(source_nodes)}")

selected_forward_nodes = database.get_nodes_by_attr(attr_values=forward_selection, attr_name="stId")
print(f"Found forward selected nodes: {len(selected_forward_nodes)}")
selected_reverse_nodes = database.get_nodes_by_attr(attr_values=reverse_selection, attr_name="stId")
print(f"Found reverse selected nodes: {len(selected_reverse_nodes)}")

Found source nodes: 10
Found forward selected nodes: 10
Found reverse selected nodes: 16


## Project graph into memory

In [ ]:

tracegraph = RadiateTrace(Reactome(database))

# set up output directory where the excel and graph files will write to
tracegraph.datadir = output_dir

# initiate tracegraph by loading graph data from arango
# a networkx graph is created here.
tracegraph.init_default_graph()

INFO: load reactome graph
INFO: MultiDirectedGraph with 71225 nodes and 112575 edges


## Perform Radiate Tracing

In [25]:


def export_radiate_traces(
    tracegraph,
    source_name,
    source_nodes,
    forward_nodes: list=None,
    reverse_nodes: list=None,
):
    """
    source_name: the data set name for radiate analysis
    source_nodes: list of source nodes used for radiate analysis
    forward_selection: dict for col_name:eids for nodes selected based on pageranks
    reverse_selection: dict for col_name:eids for nodes selected based on rev_pageranks
    """
    tracegraph.graph = tracegraph.orig_graph.copy()
    tracegraph.set_node_set_from_arango_nodes(
        source_nodes, source_name, source_name
    )

    # set pagerank or rev_pagerank property
    pagerank_prop = 'pagerank'
    rev_pagerank_prop = 'rev_pagerank'

    if forward_nodes:
        tracegraph.set_pagerank(source_name, pagerank_prop, False)

    if reverse_nodes:
        tracegraph.set_pagerank(source_name, rev_pagerank_prop, True)

    # add graph description
    tracegraph.add_graph_description('Reactome')

    # add forward traces
    if forward_nodes:
        nodeset_name = 'forward select'
        tracegraph.set_node_set_from_arango_nodes(
            forward_nodes, nodeset_name, nodeset_name
        )
            
        # add traces from sources to each selected nodes
        tracegraph.add_traces_from_sources_to_each_selected_nodes(
            forward_nodes,
            source_name,
            weighted_prop=pagerank_prop,
            selected_nodes_name=nodeset_name,
        )

        # add traces from sources to all selected nodes
        tracegraph.add_trace_from_sources_to_all_selected_nodes(
            nodeset_name,
            source_name,
            weighted_prop=pagerank_prop,
            trace_name=f'Forward combined selected nodes',
        )

    # add reverse traces
    if reverse_nodes:
        nodeset_name = 'reverse select'
        tracegraph.set_node_set_from_arango_nodes(
            reverse_nodes, nodeset_name, nodeset_name
        )

        # add traces from each selected nodes to SOURCE_SET genes
        tracegraph.add_traces_from_each_selected_nodes_to_targets(
            reverse_nodes,
            source_name,
            weighted_prop=rev_pagerank_prop,
            selected_nodes_name=nodeset_name,
        )

        # add traces from all reverse-selected nodes to SOURCE_SET
        tracegraph.add_trace_from_all_selected_nodes_to_targets(
            nodeset_name,
            source_name,
                weighted_prop=rev_pagerank_prop,
                trace_name=f"Reverse combined selected nodes",
            )
            

    # write all traces into one graph file
    graph_file = f'Radiate_traces_for_{source_name}.graph'
    tracegraph.write_to_sankey_file(graph_file)

In [26]:
export_radiate_traces(
    tracegraph,
    source_name,
    source_nodes,
    forward_nodes=selected_forward_nodes,
    reverse_nodes=selected_reverse_nodes,
    )

INFO: Adding trace network abnormal_metabolites to PYR [mitochondrial matrix] #1
ERROR: Target 194287 cannot be reached from given sources
ERROR: Target 194287 cannot be reached from given sources
ERROR: Target 194287 cannot be reached from given sources
ERROR: Target 194287 cannot be reached from given sources
ERROR: Target 194287 cannot be reached from given sources
ERROR: Target 194287 cannot be reached from given sources
INFO: Adding trace network abnormal_metabolites to AscH- [cytosol] #2
ERROR: Target 656827 cannot be reached from given sources
ERROR: Target 656827 cannot be reached from given sources
ERROR: Target 656827 cannot be reached from given sources
ERROR: Target 656827 cannot be reached from given sources
ERROR: Target 656827 cannot be reached from given sources
ERROR: Target 656827 cannot be reached from given sources
INFO: Adding trace network abnormal_metabolites to HP-HIF3A [cytosol] #3
ERROR: Target 1049793 cannot be reached from given sources
ERROR: Target 1049793